In [2]:
from apps.backend.routers.db import project_engine, engine

In [3]:
engine.dispose()

In [4]:
import types

import pandas as pd
import numpy as np
import chainladder as cl

from apps.backend.routers.methods import MethodConfigRequest


payload: MethodConfigRequest = {
    'project_id': 'proj_eb8ce71b',
    "method_type" : "chainladder",
    "source_table" : "dat_eb12fc2f",
    "config": {
        "averageMethod" : 'volume',
        'nPeriods' : 'all'
    }
}
payload = types.SimpleNamespace(**payload)


In [5]:
pe = project_engine(payload.project_id)

In [6]:
from sqlalchemy import inspect
inspector = inspect(pe)
print(inspector.get_table_names())

['dat_b1430f4b', 'dat_eb12fc2f', 'project_meta', 'project_tables']


In [7]:
df = pd.read_sql_table(payload.source_table, con=pe)
print(df.head())

   development  origin  incurred      paid           lob
0         2000    2000  325423.0  101125.0  PersonalAuto
1         2001    2001  323627.0  102541.0  PersonalAuto
2         2002    2002  358410.0  114932.0  PersonalAuto
3         2003    2003  405319.0  114452.0  PersonalAuto
4         2004    2004  434065.0  115597.0  PersonalAuto


In [8]:
# We mapped these exact column names during the ingest phase!
triangle = cl.Triangle(
    df,
    origin="origin",
    development="development",
    columns="paid",  # let this be choosable #LATER...
    cumulative=True
)
triangle

,12,24,36,48,60,72,84,96,108,120
2000,"120,952","254,370","327,823","382,505","415,929","436,364","448,283","455,982","458,780","460,274"
2001,"124,872","251,693","329,466","395,538","433,890","453,347","462,971","467,228","469,094",
2002,"137,465","272,188","363,811","433,977","469,804","490,671","499,584","503,583",,
2003,"137,580","279,089","382,614","438,732","473,128","490,763","497,672",,,
2004,"140,650","300,831","399,822","459,426","490,039","508,918",,,,
2005,"157,896","324,183","419,263","474,615","508,051",,,,,
2006,"170,380","331,419","418,440","479,344",,,,,,
2007,"158,980","307,720","410,411",,,,,,,
2008,"169,190","324,470",,,,,,,,
2009,"172,573",,,,,,,,,


In [38]:
# 3. Execute the specific method
print("chainladder" if payload.method_type == "chainladder" else "nothing")
avg_method = payload.config.get("averageMethod", "volume")
n_periods = payload.config.get("nPeriods", "all")

# Handle the 'all' string from the frontend
if n_periods == "all":
    n_periods = -1
else:
    n_periods = int(n_periods)

# Build and fit the model
dev = cl.Development(average=avg_method, n_periods=n_periods).fit(triangle)

# total_ibnr = float(fitted.ibnr_.sum().values[0][0])
# total_ultimate = float(fitted.ultimate_.sum().values[0][0])
# print(total_ibnr)

# print({
#     "status": "success",
#     "results": {"total_ibnr": total_ibnr, "projected_ultimate": total_ultimate},
# })


chainladder


In [39]:
model = cl.Chainladder()

fitted = model.fit(dev.fit_transform(triangle))

In [46]:
fitted.ibnr_.sum()

np.float64(1008716.9821425946)